# Power BI — Vistas Analíticas y Conexión

Este notebook construye **6 vistas analíticas pre-agregadas** sobre la capa Gold,
que alimentan **4 dashboards** en Power BI:

1. **Executive Overview** — KPIs ejecutivos, tendencias, mapa mundial
2. **Operational Deep Dive** — drill-down por país → destino → propiedad, análisis ABC
3. **Customer & Revenue Analytics** — segmentación RFM de usuarios, cohortes, LTV
4. **Data Quality & Pipeline Health** — retención, frescura, anomalías

Las vistas viven en el schema `gold` y se prefijan con `vw_`. Power BI se
conecta vía conector nativo (Azure Databricks) con DirectQuery o Import Mode.

## Por qué pre-agregar en vistas en lugar de medidas DAX

- El motor distribuido de Spark SQL es más eficiente para agregaciones grandes
  que el engine de Power BI.
- Cualquier herramienta (Power BI, Tableau, Excel, notebooks) ve las mismas
  cifras — no se duplican definiciones de KPIs.
- Cambios en la lógica se hacen en un solo lugar (el notebook), no en cada
  reporte.
- Reduce drásticamente el peso del modelo en Power BI.

## Parte A — Conexión de Power BI Desktop a Databricks

### Paso 1. SQL Warehouse

1. En Databricks: menú izquierdo → **SQL Warehouses** → asegurarse de tener uno corriendo (si no, **Create** → Serverless 2X-Small).
2. Abrir el warehouse → pestaña **Connection details** → copiar:
   - **Server hostname** (ej. `dbc-xxxxx.cloud.databricks.com`)
   - **HTTP path** (ej. `/sql/1.0/warehouses/abc123def`)

### Paso 2. Personal Access Token (PAT)

1. Esquina superior derecha → ícono de usuario → **Settings** → **Developer** → **Access tokens** → **Generate new token**.
2. Comment: `power-bi-wanderbricks`, Lifetime: `90 days` → **Generate**.
3. **Copiar el token completo** (no se vuelve a mostrar).

### Paso 3. Power BI Desktop

1. Descargar gratis: https://www.microsoft.com/power-platform/products/power-bi/desktop
2. **Obtener datos** → buscar **Azure Databricks** → **Conectar**.
3. Pegar Server hostname + HTTP Path → **DirectQuery** (recomendado).
4. Autenticación: **Personal Access Token** → pegar el token.
5. En el navegador, expandir → schema `gold` → seleccionar todas las vistas que empiecen con `vw_` + todas las dimensiones `gold_dim_*` → **Cargar**.

## Parte B — Construcción de las vistas analíticas

## Vista 1 — `vw_executive_kpis`

KPIs ejecutivos agregados por **país-año-mes**, con `LAG` y `SUM OVER` para
comparar contra el mes anterior y calcular running totals YTD.

In [ ]:
%sql
CREATE OR REPLACE VIEW gold.vw_executive_kpis AS
WITH base AS (
  SELECT
    t.year, t.quarter, t.month, t.month_name,
    d.country                            AS pais_destino,
    d.destination_name                   AS destino,
    COUNT(*)                             AS total_reservas,
    -- ============================================================
    -- MÉTRICAS FINANCIERAS — separación GMV vs Revenue (estándar industria)
    -- ============================================================
    -- GMV (Gross Merchandise Value): valor total reservado en la plataforma,
    -- sin importar el estado. Mide el TAMAÑO del negocio.
    -- Es lo que Airbnb / Booking / Expedia reportan como "Gross Bookings".
    SUM(f.total_amount)                  AS gmv_bruto,
    -- Revenue Neto: dinero de reservas CONFIRMADAS (las que realmente generaron ingreso).
    -- Es lo que reportan como "Revenue" en sus estados financieros.
    SUM(CASE WHEN f.booking_status = 'confirmed' THEN f.total_amount ELSE 0 END) AS revenue_neto,
    -- Revenue Perdido por cancelaciones: costo de oportunidad
    SUM(CASE WHEN f.booking_status = 'cancelled' THEN f.total_amount ELSE 0 END) AS revenue_perdido_cancelaciones,
    -- Métrica histórica mantenida para retrocompatibilidad con dashboards previos
    SUM(f.total_amount)                  AS ingresos_totales,
    -- ============================================================
    AVG(f.total_amount)                  AS ticket_promedio,
    AVG(CASE WHEN f.booking_status = 'confirmed' THEN f.total_amount END) AS ticket_promedio_confirmado,
    AVG(f.total_nights)                  AS noches_promedio,
    SUM(f.total_nights)                  AS noches_totales,
    SUM(f.payment_amount)                AS pagos_recibidos,
    COUNT(DISTINCT f.user_id)            AS usuarios_unicos,
    COUNT(DISTINCT f.property_id)        AS propiedades_reservadas,
    SUM(CASE WHEN f.booking_status = 'confirmed' THEN 1 ELSE 0 END) AS reservas_confirmadas,
    SUM(CASE WHEN f.booking_status = 'cancelled' THEN 1 ELSE 0 END) AS reservas_canceladas,
    SUM(CASE WHEN f.booking_status = 'pending'   THEN 1 ELSE 0 END) AS reservas_pendientes
  FROM gold.gold_fact_reservas f
  JOIN gold.gold_dim_time t          ON f.tiempo_id      = t.tiempo_id
  JOIN gold.gold_dim_destinations d  ON f.destination_id = d.destination_key
  GROUP BY t.year, t.quarter, t.month, t.month_name, d.country, d.destination_name
)
SELECT
  *,
  -- ============================================================
  -- Versiones redondeadas para Power BI
  -- ============================================================
  ROUND(gmv_bruto, 2)                                         AS gmv_bruto_r,
  ROUND(revenue_neto, 2)                                      AS revenue_neto_r,
  ROUND(revenue_perdido_cancelaciones, 2)                     AS revenue_perdido_r,
  ROUND(ingresos_totales, 2)                                  AS ingresos_totales_r,
  ROUND(ticket_promedio,  2)                                  AS ticket_promedio_r,
  ROUND(ticket_promedio_confirmado, 2)                        AS ticket_promedio_confirmado_r,
  ROUND(noches_promedio,  2)                                  AS noches_promedio_r,
  -- ============================================================
  -- KPIs de eficiencia operativa
  -- ============================================================
  -- Tasa de conversión: % de reservas que terminan confirmadas (benchmark industria: 75-80%)
  ROUND(reservas_confirmadas * 100.0 / total_reservas, 2)     AS tasa_conversion_pct,
  -- Tasa de cancelación (complementaria a la conversión)
  ROUND(reservas_canceladas * 100.0 / total_reservas, 2)      AS pct_cancelacion,
  -- Ratio Revenue/GMV: qué porcentaje del valor reservado se convierte en dinero real
  ROUND(revenue_neto * 100.0 / NULLIF(gmv_bruto, 0), 2)       AS ratio_revenue_gmv_pct,
  -- ============================================================
  -- Window functions analíticas (comparación temporal)
  -- ============================================================
  LAG(gmv_bruto) OVER (
    PARTITION BY pais_destino, destino ORDER BY year, month
  )                                                            AS gmv_mes_anterior,
  ROUND(
    (gmv_bruto - LAG(gmv_bruto) OVER (
      PARTITION BY pais_destino, destino ORDER BY year, month
    )) * 100.0 / NULLIF(LAG(gmv_bruto) OVER (
      PARTITION BY pais_destino, destino ORDER BY year, month
    ), 0), 2
  )                                                            AS variacion_mom_pct,
  SUM(gmv_bruto) OVER (
    PARTITION BY year, pais_destino, destino
    ORDER BY month
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  )                                                            AS gmv_ytd,
  SUM(revenue_neto) OVER (
    PARTITION BY year, pais_destino, destino
    ORDER BY month
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  )                                                            AS revenue_neto_ytd
FROM base;

In [ ]:
%sql
SELECT * FROM gold.vw_executive_kpis ORDER BY year, month, ingresos_totales DESC LIMIT 10;

## Vista 2 — `vw_operational_detail`

Detalle a nivel reserva con buckets de estancia, tipo de viaje
(doméstico vs internacional) y % pagado.

In [ ]:
%sql
CREATE OR REPLACE VIEW gold.vw_operational_detail AS
SELECT
  f.booking_id,
  f.check_in,
  f.check_out,
  f.total_nights,
  f.guests_count,
  f.total_amount,
  f.payment_amount,
  ROUND(f.payment_amount / NULLIF(f.total_amount, 0) * 100, 2) AS pct_pagado,
  f.booking_status,
  u.name                AS usuario,
  u.country             AS pais_usuario,
  u.user_type,
  u.is_business,
  p.title               AS propiedad,
  p.property_type,
  p.max_guests,
  p.bedrooms,
  p.base_price,
  d.destination_name    AS destino,
  d.country             AS pais_destino,
  d.state_or_province   AS region,
  t.year, t.quarter, t.month_name, t.day_name, t.is_weekend,
  CASE
    WHEN u.country = d.country THEN 'doméstico'
    ELSE 'internacional'
  END                   AS tipo_viaje,
  CASE
    WHEN f.total_nights <= 2  THEN '1-2 noches'
    WHEN f.total_nights <= 7  THEN '3-7 noches'
    WHEN f.total_nights <= 14 THEN '8-14 noches'
    ELSE '15+ noches'
  END                   AS bucket_estancia
FROM gold.gold_fact_reservas f
LEFT JOIN gold.gold_dim_users        u ON f.user_id        = u.user_key
LEFT JOIN gold.gold_dim_properties   p ON f.property_id    = p.property_key
LEFT JOIN gold.gold_dim_destinations d ON f.destination_id = d.destination_key
LEFT JOIN gold.gold_dim_time         t ON f.tiempo_id      = t.tiempo_id;

In [ ]:
%sql
SELECT * FROM gold.vw_operational_detail ORDER BY check_in DESC LIMIT 10;

## Vista 3 — `vw_customer_analytics`

Métricas por usuario: LTV, número de reservas, recencia, y segmentación RFM
(Recency–Frequency–Monetary) con etiquetas accionables tipo "Champions",
"At Risk", "Hibernating".

In [ ]:
%sql
CREATE OR REPLACE VIEW gold.vw_customer_analytics AS
WITH metricas AS (
  SELECT
    u.user_key                       AS user_id,
    u.name,
    u.country                        AS pais_usuario,
    u.user_type,
    u.is_business,
    u.registration_date,
    COUNT(*)                         AS num_reservas,
    SUM(f.total_amount)              AS ltv_total,
    AVG(f.total_amount)              AS ticket_promedio,
    SUM(f.total_nights)              AS noches_totales,
    MIN(f.check_in)                  AS primera_reserva,
    MAX(f.check_in)                  AS ultima_reserva,
    COUNT(DISTINCT f.destination_id) AS destinos_visitados,
    SUM(CASE WHEN f.booking_status = 'cancelled' THEN 1 ELSE 0 END) AS canceladas
  FROM gold.gold_dim_users u
  JOIN gold.gold_fact_reservas f ON u.user_key = f.user_id
  GROUP BY u.user_key, u.name, u.country, u.user_type, u.is_business, u.registration_date
),
rfm AS (
  SELECT
    *,
    DATEDIFF(CURRENT_DATE(), ultima_reserva)                  AS dias_desde_ultima,
    NTILE(5) OVER (ORDER BY DATEDIFF(CURRENT_DATE(), ultima_reserva) ASC) AS r_score,
    NTILE(5) OVER (ORDER BY num_reservas ASC)                 AS f_score,
    NTILE(5) OVER (ORDER BY ltv_total ASC)                    AS m_score
  FROM metricas
)
SELECT
  *,
  CASE
    WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4 THEN 'Champions'
    WHEN r_score >= 3 AND m_score >= 4                  THEN 'Big Spenders'
    WHEN r_score >= 4 AND f_score >= 3                  THEN 'Loyal'
    WHEN r_score <= 2 AND m_score >= 4                  THEN 'At Risk (high value)'
    WHEN r_score <= 2                                    THEN 'Hibernating'
    ELSE 'Regular'
  END                                                  AS segmento_rfm,
  ROUND(ltv_total, 2)                                  AS ltv_total_r,
  ROUND(ticket_promedio, 2)                            AS ticket_promedio_r
FROM rfm;

In [ ]:
%sql
SELECT segmento_rfm, COUNT(*) AS usuarios, ROUND(SUM(ltv_total), 0) AS ltv_segmento
FROM gold.vw_customer_analytics
GROUP BY segmento_rfm
ORDER BY ltv_segmento DESC;

## Vista 4 — `vw_property_performance`

Ranking de propiedades por ingresos con clasificación ABC tipo Pareto
(A = 70% de ingresos, B = siguiente 20%, C = resto 10%).

In [ ]:
%sql
CREATE OR REPLACE VIEW gold.vw_property_performance AS
WITH base AS (
  SELECT
    p.property_key       AS property_id,
    p.title,
    p.property_type,
    p.base_price,
    p.max_guests,
    d.destination_name   AS destino,
    d.country            AS pais_destino,
    COUNT(*)                        AS num_reservas,
    SUM(f.total_amount)             AS ingresos,
    SUM(f.total_nights)             AS noches_totales,
    AVG(f.total_amount)             AS ticket_promedio,
    AVG(f.total_nights)             AS noches_promedio,
    COUNT(DISTINCT f.user_id)       AS huespedes_unicos
  FROM gold.gold_dim_properties p
  JOIN gold.gold_fact_reservas f   ON p.property_key   = f.property_id
  LEFT JOIN gold.gold_dim_destinations d ON p.destination_id = d.destination_key
  GROUP BY p.property_key, p.title, p.property_type, p.base_price, p.max_guests,
           d.destination_name, d.country
),
ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (ORDER BY ingresos DESC)         AS rank_ingresos,
    SUM(ingresos) OVER ()                              AS ingresos_total_red,
    SUM(ingresos) OVER (ORDER BY ingresos DESC
                        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS ingresos_acum
  FROM base
)
SELECT
  *,
  ROUND(ingresos * 100.0 / ingresos_total_red, 4)      AS pct_ingresos,
  ROUND(ingresos_acum * 100.0 / ingresos_total_red, 2) AS pct_acumulado,
  CASE
    WHEN ingresos_acum * 100.0 / ingresos_total_red <= 70  THEN 'A (top 70%)'
    WHEN ingresos_acum * 100.0 / ingresos_total_red <= 90  THEN 'B (siguiente 20%)'
    ELSE 'C (resto 10%)'
  END                                                  AS clasificacion_abc
FROM ranked;

In [ ]:
%sql
SELECT clasificacion_abc, COUNT(*) AS propiedades, ROUND(SUM(pct_ingresos), 2) AS pct_ingresos_clase
FROM gold.vw_property_performance
GROUP BY clasificacion_abc
ORDER BY clasificacion_abc;

## Vista 5 — `vw_data_quality` y `vw_data_quality_detail`

Métricas de salud del pipeline.

In [ ]:
%sql
CREATE OR REPLACE VIEW gold.vw_data_quality AS
SELECT 'bookings' AS entidad,
  (SELECT COUNT(*) FROM bronze.bronze_bookings)     AS registros_bronze,
  (SELECT COUNT(*) FROM silver.silver_bookings)     AS registros_silver,
  ROUND((SELECT COUNT(*) FROM silver.silver_bookings) * 100.0 / NULLIF((SELECT COUNT(*) FROM bronze.bronze_bookings), 0), 2) AS pct_retenido
UNION ALL
SELECT 'users', (SELECT COUNT(*) FROM bronze.bronze_users), (SELECT COUNT(*) FROM silver.silver_users),
  ROUND((SELECT COUNT(*) FROM silver.silver_users) * 100.0 / NULLIF((SELECT COUNT(*) FROM bronze.bronze_users), 0), 2)
UNION ALL
SELECT 'properties', (SELECT COUNT(*) FROM bronze.bronze_properties), (SELECT COUNT(*) FROM silver.silver_properties),
  ROUND((SELECT COUNT(*) FROM silver.silver_properties) * 100.0 / NULLIF((SELECT COUNT(*) FROM bronze.bronze_properties), 0), 2)
UNION ALL
SELECT 'payments', (SELECT COUNT(*) FROM bronze.bronze_payments), (SELECT COUNT(*) FROM silver.silver_payments),
  ROUND((SELECT COUNT(*) FROM silver.silver_payments) * 100.0 / NULLIF((SELECT COUNT(*) FROM bronze.bronze_payments), 0), 2)
UNION ALL
SELECT 'reviews', (SELECT COUNT(*) FROM bronze.bronze_reviews), (SELECT COUNT(*) FROM silver.silver_reviews),
  ROUND((SELECT COUNT(*) FROM silver.silver_reviews) * 100.0 / NULLIF((SELECT COUNT(*) FROM bronze.bronze_reviews), 0), 2)
UNION ALL
SELECT 'destinations', (SELECT COUNT(*) FROM bronze.bronze_destinations), (SELECT COUNT(*) FROM silver.silver_destinations),
  ROUND((SELECT COUNT(*) FROM silver.silver_destinations) * 100.0 / NULLIF((SELECT COUNT(*) FROM bronze.bronze_destinations), 0), 2);

In [ ]:
%sql
CREATE OR REPLACE VIEW gold.vw_data_quality_detail AS
SELECT
  MIN(check_in)                                  AS fecha_minima,
  MAX(check_in)                                  AS fecha_maxima,
  DATEDIFF(MAX(check_in), MIN(check_in))         AS rango_dias,
  COUNT(*)                                       AS total_reservas,
  COUNT(CASE WHEN payment_amount = 0 THEN 1 END) AS reservas_sin_pago,
  ROUND(COUNT(CASE WHEN payment_amount = 0 THEN 1 END) * 100.0 / COUNT(*), 2) AS pct_sin_pago,
  COUNT(CASE WHEN total_amount > 5000 THEN 1 END) AS reservas_alto_monto,
  COUNT(CASE WHEN total_nights > 30 THEN 1 END)   AS reservas_estancia_larga,
  COUNT(CASE WHEN booking_status = 'cancelled' THEN 1 END) AS reservas_canceladas,
  ROUND(COUNT(CASE WHEN booking_status = 'cancelled' THEN 1 END) * 100.0 / COUNT(*), 2) AS pct_cancelacion
FROM gold.gold_fact_reservas;

In [ ]:
%sql
SELECT * FROM gold.vw_data_quality ORDER BY entidad;


## Vista 6 — `vw_cohort_analysis`

Análisis de cohortes: usuarios agrupados por mes de registro y comportamiento
en meses posteriores. Estándar de growth analytics.

In [ ]:
%sql
CREATE OR REPLACE VIEW gold.vw_cohort_analysis AS
WITH cohortes AS (
  SELECT
    u.user_key                                       AS user_id,
    DATE_FORMAT(u.registration_date, 'yyyy-MM')      AS cohorte_registro,
    DATE_FORMAT(f.check_in, 'yyyy-MM')               AS mes_actividad,
    f.total_amount
  FROM gold.gold_dim_users u
  JOIN gold.gold_fact_reservas f ON u.user_key = f.user_id
)
SELECT
  cohorte_registro,
  mes_actividad,
  COUNT(DISTINCT user_id)         AS usuarios_activos,
  COUNT(*)                        AS reservas,
  ROUND(SUM(total_amount), 2)     AS ingresos
FROM cohortes
GROUP BY cohorte_registro, mes_actividad
ORDER BY cohorte_registro, mes_actividad;

In [ ]:
%sql
SELECT * FROM gold.vw_cohort_analysis LIMIT 20;

## Validación final

In [ ]:
%sql
SHOW VIEWS IN gold LIKE 'vw_*';

## Parte C — Guía de construcción de los 4 dashboards en Power BI

La configuración detallada de cada visualización (campos exactos, tipo de gráfico,
formato, paleta de colores, filtros, mockups visuales) está en:

**→ [`documentation/dashboards_powerbi.md`](../documentation/dashboards_powerbi.md)**

Ábrelo en paralelo con Power BI Desktop y sigue las instrucciones visual por visual.

## Resumen

| Vista | Dashboard que alimenta |
|---|---|
| `vw_executive_kpis` | Dashboard 1 — Executive Overview |
| `vw_operational_detail` | Dashboard 2 — Operational Deep Dive |
| `vw_property_performance` | Dashboard 2 — Operational Deep Dive (análisis ABC) |
| `vw_customer_analytics` | Dashboard 3 — Customer & Revenue |
| `vw_cohort_analysis` | Dashboard 3 — Customer & Revenue (cohortes) |
| `vw_data_quality` + `vw_data_quality_detail` | Dashboard 4 — Data Quality & Pipeline Health |